### Creating new variables into the dataset (with DFS)

In [30]:
import pandas as pd
import numpy as np

# pip install featuretools
import featuretools as ft

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression
from sklearn.model_selection import train_test_split


# load data
df = pd.read_csv("winequality-red.csv")
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5


**DFS - automate generating new variables by BRUTE FORCE**

By default, you should drop the target variable, because you can't have the target variable to be in any part in any new variable, because that is technically target leakage. (not realistic from an actual ML model point of view)

In [31]:
# define the target variable
target = "quality"

# save for later use
y = df[target]

# create an entity set with the red wine data
es = ft.EntitySet(id="red_wine_quality")
es = es.add_dataframe(dataframe_name="red_wine_data", 
                      dataframe=df.drop(target, axis=1),
                      index="index",
                      make_index=True)


# Use DFS to generate new variables (features), aggregate BASED on target variable: quality
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="red_wine_data",
    
    # agg_primitives=[],
    # if you use agg primitives, define also the calculations:
    # you can also use groupby trans primitives to nominal categories
    # groupby_trans_primitives=["cum_mean"],
    #trans_primitives=["multiply_numeric"],
    trans_primitives=["divide_numeric", "multiply_numeric"],
    # increase depth if you want to capture more complex
    # relationships in the data
    max_depth=1
)

# division a bit problematic, since often times
# values we divide with something very small (but not 0)
# this results in many rows and columns to go into infinity
# we have to take care of this

# THIS PART ONLY REQUIRED IF YOU DO DIVISION
# IT REMOVES ALL COLUMNS WITH INFINITE VALUES
feature_matrix.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_matrix.dropna(axis=1, inplace=True)

# re-attach y back to feature_matrix
feature_matrix[target] = y

feature_matrix

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,...,pH * sulphates,pH * total sulfur dioxide,pH * volatile acidity,residual sugar * sulphates,residual sugar * total sulfur dioxide,residual sugar * volatile acidity,sulphates * total sulfur dioxide,sulphates * volatile acidity,total sulfur dioxide * volatile acidity,quality
index,,,,,,,,,,,,,,,,,,,,,
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,...,1.9656,119.34,2.45700,1.064,64.6,1.330,19.04,0.39200,23.80,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,...,2.1760,214.40,2.81600,1.768,174.2,2.288,45.56,0.59840,58.96,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,...,2.1190,176.04,2.47760,1.495,124.2,1.748,35.10,0.49400,41.04,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,...,1.8328,189.60,0.88480,1.102,114.0,0.532,34.80,0.16240,16.80,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,...,1.9656,119.34,2.45700,1.064,64.6,1.330,19.04,0.39200,23.80,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1594,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,...,2.0010,151.80,2.07000,1.160,88.0,1.200,25.52,0.34800,26.40,5
1595,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,...,2.6752,179.52,1.93600,1.672,112.2,1.210,38.76,0.41800,28.05,6
1596,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,...,2.5650,136.80,1.74420,1.725,92.0,1.173,30.00,0.38250,20.40,6


In [32]:
# typical X/y -split
X = feature_matrix.drop("quality", axis=1)
y = feature_matrix['quality']

# define model (linear regression in this example)
# technically you can use pretty much any classic ML algorithm
model = LinearRegression()
# model = RandomForestRegressor()

# create RFE, place the model and choose amount of optimal variables
rfe = RFE(estimator=model, n_features_to_select=12)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and the results
rankings = rfe.ranking_
support = rfe.support_

results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# if having lots of new variables, use Data Wrangler or
# something else to see all results
results_df.iloc[0:15]

,Feature,Ranking,Selected
15,alcohol / pH,1,True
45,density / sulphates,1,True
102,volatile acidity / alcohol,1,True
113,alcohol * density,1,True
118,alcohol * sulphates,1,True
99,total sulfur dioxide / residual sugar,1,True
92,sulphates / volatile acidity,1,True
66,pH / alcohol,1,True
97,total sulfur dioxide / free sulfur dioxide,1,True
150,fixed acidity * volatile acidity,1,True


<b>It might be a good idea to go through the redundancy of the new variables before deciding the optimal ones, since RFE (for example) might suggest many variables with high multicollinearity. (e.g. chlorides, density and "chlorides * density")</b>

<b>Another idea: Can we use newly generated highly ranked variables in order to reduce dimensionality? For example, retain the new column "volatile acidity / density" and remove the original volatile acidity and density? Is this viable?</b>

Answer: It's usually recommended to remove the original variables if you choose a combination variable to replace them. BUT! Always be aware you might lose hidden connections or destroy the authenticity of the dataset.

Always inspect statistics, correlations / associations, compare ML model performance before and after, always test on REAL DATA ONLY (if you use synthetization). In other words, any new combination variable should "make sense" from the point of view of the dataset optimization and the final ML model performance in practice.

In other other words: this is all about experimentation. If you can, find a good reason why to use a certain method, and if you can't find a clear reason, inspect all possible metrics / visualizations and finally compare your ML model performance before and after an optimization in order to see if your model went better or worse.